This analysis looks at how school resources, class size, and teacher qualifications are related to mathematics performance in seven education systems (Germany, France, Poland, Estonia, Italy, Spain and Ukraine, the latter covering 18 of 27 regions).

RQ1 studies these school factors. RQ2 adds students' socioeconomic background (ESCS) to see whether the results change.

In [1]:
from pathlib import Path
import math, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_FILE = PROJECT_ROOT / "data/processed/pisa_cleaned_dataset.parquet"
if not DATA_FILE.exists():
    raise FileNotFoundError(f"Missing {DATA_FILE.resolve()}")

# Team-wide country selection (same 7 systems as RQ3/RQ4)
PROJECT_COUNTRIES = ["DEU", "FRA", "POL", "EST", "ITA", "ESP", "QUR"]

df_full = pd.read_parquet(DATA_FILE, engine="pyarrow")
df = df_full[df_full["CNT"].isin(PROJECT_COUNTRIES)].reset_index(drop=True)
print(f"Full file: {len(df_full):,} records. Project subset: {len(df):,} records, "
      f"{df['CNT'].nunique()} systems.")
display(df.head(3))

Full file: 121,564 records. Project subset: 70,517 records, 7 systems.


,CNT,CNTSCHID,ESCS,W_FSTUWT,W_FSTURWT1,W_FSTURWT2,W_FSTURWT3,W_FSTURWT4,W_FSTURWT5,W_FSTURWT6,W_FSTURWT7,W_FSTURWT8,W_FSTURWT9,W_FSTURWT10,W_FSTURWT11,W_FSTURWT12,W_FSTURWT13,W_FSTURWT14,W_FSTURWT15,W_FSTURWT16,W_FSTURWT17,W_FSTURWT18,W_FSTURWT19,W_FSTURWT20,W_FSTURWT21,W_FSTURWT22,W_FSTURWT23,W_FSTURWT24,W_FSTURWT25,W_FSTURWT26,W_FSTURWT27,W_FSTURWT28,W_FSTURWT29,W_FSTURWT30,W_FSTURWT31,W_FSTURWT32,W_FSTURWT33,W_FSTURWT34,W_FSTURWT35,W_FSTURWT36,W_FSTURWT37,W_FSTURWT38,W_FSTURWT39,W_FSTURWT40,W_FSTURWT41,W_FSTURWT42,W_FSTURWT43,W_FSTURWT44,W_FSTURWT45,W_FSTURWT46,W_FSTURWT47,W_FSTURWT48,W_FSTURWT49,W_FSTURWT50,W_FSTURWT51,W_FSTURWT52,W_FSTURWT53,W_FSTURWT54,W_FSTURWT55,W_FSTURWT56,W_FSTURWT57,W_FSTURWT58,W_FSTURWT59,W_FSTURWT60,W_FSTURWT61,W_FSTURWT62,W_FSTURWT63,W_FSTURWT64,W_FSTURWT65,W_FSTURWT66,W_FSTURWT67,W_FSTURWT68,W_FSTURWT69,W_FSTURWT70,W_FSTURWT71,W_FSTURWT72,W_FSTURWT73,W_FSTURWT74,W_FSTURWT75,W_FSTURWT76,W_FSTURWT77,W_FSTURWT78,W_FSTURWT79,W_FSTURWT80,PV1MATH,PV2MATH,PV3MATH,PV4MATH,PV5MATH,PV6MATH,PV7MATH,PV8MATH,PV9MATH,PV10MATH,ESCS_imputed,SCHLTYPE,PROATCE,STRATIO,STAFFSHORT,EDUSHORT,CLSIZE,SCH_ESCS,STRATIO_imputed,PROATCE_imputed,EDUSHORT_imputed,STAFFSHORT_imputed,CLSIZE_imputed,SCH_ESCS_imputed,SCHLTYPE_imputed,SENWT,SCHOOL_GROUP_ID
0,EST,23300149.0,0.9668,4.69717,7.14360,7.14360,7.14360,7.14360,7.14360,7.14360,7.14360,7.14360,7.14360,7.14360,7.14360,7.14360,7.14360,7.14360,7.14360,7.14360,7.14360,7.14360,7.14360,7.14360,7.14360,7.14360,7.14360,7.14360,7.14360,7.14360,7.14360,7.14360,7.14360,7.14360,7.14360,7.14360,7.14360,7.14360,7.14360,7.14360,7.14360,7.14360,7.14360,7.14360,2.30841,2.30841,2.30841,2.30841,2.30841,2.30841,2.30841,2.30841,2.30841,2.30841,2.30841,2.30841,2.30841,2.30841,2.30841,2.30841,2.30841,2.30841,2.30841,2.30841,2.30841,2.30841,2.30841,2.30841,2.30841,2.30841,2.30841,2.30841,2.30841,2.30841,2.30841,2.30841,2.30841,2.30841,2.30841,2.30841,2.30841,2.30841,2.30841,2.30841,527.654,590.326,563.005,549.312,546.229,590.066,553.731,521.827,511.495,562.517,0,3.0,0.8667,8.1333,0.2833,-0.6483,18.0,-0.435886,0,0,0,1,0,0,0,0.351981,EST_23300149
1,EST,23300171.0,-0.3947,1.72897,1.01847,2.30472,1.01847,2.30472,1.01847,1.01847,1.01847,1.01847,2.30472,2.30472,1.01847,2.30472,2.30472,1.01847,1.01847,2.30472,2.30472,2.30472,2.30472,1.01847,1.01847,2.30472,1.01847,2.30472,1.01847,1.01847,1.01847,1.01847,2.30472,2.30472,1.01847,2.30472,2.30472,1.01847,1.01847,2.30472,2.30472,2.30472,2.30472,1.01847,2.30472,1.01847,2.30472,1.01847,2.30472,2.30472,2.30472,2.30472,1.01847,1.01847,2.30472,1.01847,1.01847,2.30472,2.30472,1.01847,1.01847,1.01847,1.01847,2.30472,2.30472,1.01847,2.30472,1.01847,2.30472,2.30472,2.30472,2.30472,1.01847,1.01847,2.30472,1.01847,1.01847,2.30472,2.30472,1.01847,1.01847,1.01847,1.01847,2.30472,715.893,719.525,738.151,742.997,732.032,697.641,687.527,695.576,679.676,677.391,0,2.0,0.7877,8.1899,0.2833,-0.9490,18.0,1.010037,0,0,0,1,0,0,0,0.129560,EST_23300171
2,EST,23300075.0,-0.9071,1.76996,2.69649,2.70108,0.90036,0.86966,0.87013,0.86966,2.61040,0.89883,2.60897,0.89883,2.60897,2.70108,2.70108,2.69649,0.90036,0.87013,2.60897,0.89883,0.87013,2.61040,2.60897,2.61040,0.87013,0.89883,0.90036,0.89883,2.70108,0.86966,2.69649,0.86966,2.69649,2.61040,2.61040,2.60897,0.87013,0.90036,2.69649,0.86966,0.90036,2.70108,2.70097,2.70589,0.90196,0.87012,0.87063,0.87012,2.61190,0.90032,2.61037,0.90032,2.61037,2.70589,2.70589,2.70097,0.90196,0.87063,2.61037,0.90032,0.87063,2.61190,2.61037,2.61190,0.87063,0.90032,0.90196,0.90032,2.70589,0.87012,2.70097,0.87012,2.70097,2.61190,2.61190,2.61037,0.87063,0.90196,2.70097,0.87012,0.90196,2.70589,433.490,420.050,406.021,428.302,405.179,448.518,451.096,426.500,396.945,375.990,0,3.0,0.9727,15.8909,0.2833,1.2435,23.0,0.444450,0,0,0,1,0,0,0,0.132631,EST_23300075


In [2]:
PV_COLS = [f"PV{i}MATH" for i in range(1, 11)]
REP_COLS = [f"W_FSTURWT{i}" for i in range(1, 81)]
PREDICTORS = ["EDUSHORT", "CLSIZE", "PROATCE"]
CONTROL = "ESCS"
AUDIT = ["CNT", "CNTSCHID", "SCHOOL_GROUP_ID", "SCH_ESCS", "SCHLTYPE", "SENWT"]
FLAGS = [f"{v}_imputed" for v in ["ESCS", "EDUSHORT", "CLSIZE", "PROATCE", "SCH_ESCS"]]
REQUIRED = ["CNT", "W_FSTUWT", CONTROL] + PREDICTORS + PV_COLS + REP_COLS
missing = [v for v in REQUIRED if v not in df.columns]
if missing:
    raise KeyError(f"Dataset lacks required variables: {missing}")

print("Available audit fields:", [x for x in AUDIT if x in df.columns])
print("Education-system codes and record counts:")
display(df["CNT"].value_counts(dropna=False).rename_axis("CNT").to_frame("students"))

inspect_cols = [CONTROL] + PREDICTORS + ["SCH_ESCS", "W_FSTUWT", "PV1MATH"]
inspect_cols = [v for v in inspect_cols if v in df.columns]
display(df[inspect_cols].describe().T)
display(df[inspect_cols].isna().mean().rename("fraction_missing").to_frame())

impute_cols = [v for v in FLAGS if v in df.columns]
if impute_cols:
    imputation_audit = df[impute_cols].apply(pd.to_numeric, errors="coerce").mean().rename("fraction_imputed")
    display(imputation_audit.to_frame())
else:
    print("No imputation indicator columns found.")

if "SCHOOL_GROUP_ID" in df.columns:
    print("Distinct school group IDs:", df["SCHOOL_GROUP_ID"].nunique(dropna=True))

Available audit fields: ['CNT', 'CNTSCHID', 'SCHOOL_GROUP_ID', 'SCH_ESCS', 'SCHLTYPE', 'SENWT']
Education-system codes and record counts:
Distinct school group IDs: 2449


,students
CNT,
ESP,30800
ITA,10552
FRA,6770
EST,6392
DEU,6116
POL,6011
QUR,3876


,count,mean,std,min,25%,50%,75%,max
ESCS,70517.0,-0.022862,0.925397,-6.58540,-0.619900,0.044400,0.716700,3.94300
EDUSHORT,70517.0,-0.205308,0.970370,-1.42120,-1.421200,-0.269200,0.429200,2.95950
CLSIZE,70517.0,25.370492,6.994695,13.00000,23.000000,23.000000,28.000000,53.00000
PROATCE,70517.0,0.828626,0.302776,0.00000,0.815400,0.988900,1.000000,1.00000
SCH_ESCS,70517.0,-0.030479,0.487052,-3.17810,-0.332857,-0.004902,0.295265,1.55235
W_FSTUWT,70517.0,41.670464,44.481648,1.02381,4.643240,23.834630,69.109820,318.40510
PV1MATH,70517.0,480.624529,88.534347,129.43400,419.237000,481.566000,542.332000,802.27600


,fraction_missing
ESCS,0.0
EDUSHORT,0.0
CLSIZE,0.0
PROATCE,0.0
SCH_ESCS,0.0
W_FSTUWT,0.0
PV1MATH,0.0


,fraction_imputed
ESCS_imputed,0.038161
EDUSHORT_imputed,0.089567
CLSIZE_imputed,0.088503
PROATCE_imputed,0.153339
SCH_ESCS_imputed,0.001617


We check that the variables needed for our analysis are available. We also examine missing values, country coverage, and which values were filled during data cleaning.

In [3]:

# DATA VERIFICATION: PISA YEAR, COUNTRIES AND SCHOOL ESCS

print("=== DATASET VERIFICATION ===")

# 1. year/cycle variable availability
year_columns = [
    c for c in ["CYS", "CYC", "YEAR", "year"]
    if c in df.columns
]

print("\nYear/cycle columns:", year_columns)

if year_columns:
    for col in year_columns:
        print(f"\nValues in {col}:")
        display(df[col].value_counts(dropna=False))
else:
    print(
        "No year/cycle variable retained. "
        "Original PISA source files must be verified."
    )

# 2. country coverage
print("\n=== COUNTRY COVERAGE ===")

country_summary = (
    df["CNT"]
    .value_counts()
    .rename_axis("Country")
    .reset_index(name="Students")
)

country_summary["Percentage"] = (
    country_summary["Students"] / len(df) * 100
)

display(country_summary.round(2))

print("Total countries/systems:", df["CNT"].nunique())
print("Total students:", len(df))

# 3. school-level socioeconomic data
print("\n=== SCHOOL ESCS VERIFICATION ===")

required_school_cols = [
    "CNT", "CNTSCHID", "SCH_ESCS"
]

if all(c in df.columns for c in required_school_cols):

    school_groups = df.groupby(
        ["CNT", "CNTSCHID"],
        dropna=False
    )

    school_escs_counts = school_groups[
        "SCH_ESCS"
    ].nunique(dropna=False)

    inconsistent_schools = (
        school_escs_counts > 1
    ).sum()

    print("Number of schools:", len(school_escs_counts))
    print(
        "Schools with inconsistent SCH_ESCS:",
        inconsistent_schools
    )

    print(
        "Missing SCH_ESCS values:",
        df["SCH_ESCS"].isna().sum()
    )

    if inconsistent_schools == 0:
        print(
            "SCH_ESCS is consistent within each school."
        )
    else:
        print(
            "WARNING: Some schools have different "
            "SCH_ESCS values for their students."
        )

else:
    print(
        "Some school identifier or ESCS columns "
        "are unavailable."
    )

print("\nData verification completed.")


=== DATASET VERIFICATION ===

Year/cycle columns: []
No year/cycle variable retained. Original PISA source files must be verified.

=== COUNTRY COVERAGE ===
Total countries/systems: 7
Total students: 70517

=== SCHOOL ESCS VERIFICATION ===
Number of schools: 2449
Schools with inconsistent SCH_ESCS: 0
Missing SCH_ESCS values: 0
SCH_ESCS is consistent within each school.

Data verification completed.


,Country,Students,Percentage
0,ESP,30800,43.68
1,ITA,10552,14.96
2,FRA,6770,9.60
3,EST,6392,9.06
4,DEU,6116,8.67
5,POL,6011,8.52
6,QUR,3876,5.50


The seven systems were chosen jointly by the team so that RQ1–RQ4 describe the same sample. Spain contributes a large share of records because of its regional oversampling, which the survey weights correct.After applying the survey weights, Spain's share falls from 43.7% of records to 15.6% of the weighted population, which is why the weights matter here.

In [4]:
# A proportion from 0 to 1 and a percentage from 0 to 100
# are both converted to number of 10-percentage-point increments.
cert = pd.to_numeric(df["PROATCE"], errors="coerce")
cert_min, cert_max = cert.min(), cert.max()
print("PROATCE range:", cert_min, "to", cert_max)
if 0 <= cert_min and cert_max <= 1.00001:
    cert_tenpp = cert * 10
    certification_scale = "fraction (0–1) converted to 10 percentage-point units"
elif 0 <= cert_min and cert_max <= 100:
    cert_tenpp = cert / 10
    certification_scale = "percentage (0–100) converted to 10 percentage-point units"
else:
    raise ValueError("Unexpected PROATCE range: check OECD codebook and cleaning steps.")
print("Teacher certification transformation:", certification_scale)

model_vars = REQUIRED + ["PROATCE_10PP"]
work = df[REQUIRED].copy()
work["PROATCE_10PP"] = cert_tenpp
for var in ["W_FSTUWT", CONTROL, "EDUSHORT", "CLSIZE", "PROATCE_10PP"] + PV_COLS + REP_COLS:
    work[var] = pd.to_numeric(work[var], errors="coerce")

num_cols = ["W_FSTUWT", CONTROL, "EDUSHORT", "CLSIZE", "PROATCE_10PP"] + PV_COLS + REP_COLS
complete = np.isfinite(work[num_cols].to_numpy(dtype=np.float64)).all(axis=1)
complete &= work["CNT"].notna().to_numpy()
complete &= (work["W_FSTUWT"] > 0).to_numpy()
complete &= (work[REP_COLS] > 0).all(axis=1).to_numpy()

work = work.loc[complete].reset_index(drop=True)
if work.empty:
    raise ValueError("No complete observations with positive survey weights remain.")

print(f"Analysis sample: {len(work):,} of {len(df):,} records ({100*len(work)/len(df):.1f}%).")
print(f"Systems retained: {work['CNT'].nunique()}.")
display(work["CNT"].value_counts().rename("students"))
print("Predictor correlation matrix (unweighted diagnostic, not an inference result):")
display(work[["EDUSHORT", "CLSIZE", "PROATCE_10PP", "ESCS"]].corr().round(3))


PROATCE range: 0.0 to 1.0
Teacher certification transformation: fraction (0–1) converted to 10 percentage-point units
Analysis sample: 70,517 of 70,517 records (100.0%).
Systems retained: 7.
Predictor correlation matrix (unweighted diagnostic, not an inference result):


,students
CNT,
ESP,30800
ITA,10552
FRA,6770
EST,6392
DEU,6116
POL,6011
QUR,3876


,EDUSHORT,CLSIZE,PROATCE_10PP,ESCS
EDUSHORT,1.000,-0.047,0.004,-0.080
CLSIZE,-0.047,1.000,0.072,0.074
PROATCE_10PP,0.004,0.072,1.000,0.030
ESCS,-0.080,0.074,0.030,1.000


We use educational-material shortages (EDUSHORT), class size (CLSIZE), and teacher certification (PROATCE) as predictors. These stand in for the three things a ministry could fund: equipment, smaller classes and better-qualified teachers.

Teacher certification is converted into units of 10 percentage points. Both regression models use the same 70,517 students from seven education systems.

In [5]:
WEIGHTING = "equal_country"  # Alternative: "population"
if WEIGHTING not in {"equal_country", "population"}:
    raise ValueError("WEIGHTING must be 'equal_country' or 'population'")

weight = work["W_FSTUWT"].to_numpy(dtype=np.float64)
if WEIGHTING == "equal_country":
    base_country_total = work.groupby("CNT")["W_FSTUWT"].transform("sum").to_numpy(dtype=np.float64)
    factors = 1.0 / base_country_total
else:
    factors = np.ones(len(work), dtype=np.float64)

w_main = weight * factors
w_rep = work[REP_COLS].to_numpy(dtype=np.float64) * factors[:, None]
print("Weighting:", WEIGHTING)
print("Sum of weights:", round(w_main.sum(), 4))
assert np.isfinite(w_main).all() and (w_main > 0).all()
assert np.isfinite(w_rep).all() and (w_rep > 0).all()

# Design matrix for common analysis sample; reference country is alphabetically first.
country_dummies = pd.get_dummies(work["CNT"].astype(str), prefix="CNT", drop_first=True, dtype=float)
countries = sorted(work["CNT"].astype(str).unique())
print("Reference system:", countries[0])

Y = work[PV_COLS].to_numpy(dtype=np.float64)
BASE_TERMS = ["EDUSHORT", "CLSIZE", "PROATCE_10PP"]

def make_design(terms):
    x = work[terms].reset_index(drop=True).astype(float)
    x = pd.concat([x, country_dummies.reset_index(drop=True)], axis=1)
    x.insert(0, "Intercept", 1.0)
    X = x.to_numpy(dtype=np.float64)
    if np.linalg.matrix_rank(X) < X.shape[1]:
        raise ValueError("Collinear or constant regressors; inspect predictors and country coverage.")
    return X, x.columns.tolist()

X1, terms1 = make_design(BASE_TERMS)
X2, terms2 = make_design(BASE_TERMS + ["ESCS"])
print("RQ1 design:", X1.shape, "RQ2 design:", X2.shape)

Weighting: equal_country
Sum of weights: 7.0
Reference system: DEU
RQ1 design: (70517, 10) RQ2 design: (70517, 11)


PISA students have different sampling weights. We use these weights to make the analysis more representative.

We also adjust the weights so that each of the 7 education systems has equal influence on the main results (14.3% each). This defines an equal-system target rather than a population-weighted target; the raw record share is not the survey-weighted share. Country differences are included in both models as fixed effects, so the estimates compare students within the same system.

In [6]:
def weighted_coefficients(X, Y, weights):
    """Solve weighted linear regressions for all mathematics PVs at once.

    X: (n_students, n_predictors); Y: (n_students, 10).
    Return (n_predictors, 10), no enormous diagonal weight matrix.
    """
    weights = np.asarray(weights, dtype=np.float64)
    if (weights <= 0).any() or not np.isfinite(weights).all():
        raise ValueError("Weights must be finite and positive")
    xtw = X.T * weights
    return np.linalg.solve(xtw @ X, xtw @ Y)


def survey_pv_regression(X, term_names, Y, w_main, w_rep, country_codes, label):
    """Estimate each PV and combine independent-country Fay-BRR sampling variance.

    Each replicate changes weights for ONE country, preserving the full weights
    elsewhere. Each system contributes a separate sampling variance term.
    Plausible-value uncertainty is combined using the standard MI formula.
    """
    m, R = Y.shape[1], w_rep.shape[1]
    if (m, R) != (10, 80):
        raise ValueError(f"Expected 10 mathematics PVs and 80 replicate weights; got {m}, {R}")
    tic = time.perf_counter()
    full_beta = weighted_coefficients(X, Y, w_main)  # predictors × PVs
    p = X.shape[1]
    sample_var_by_pv = np.zeros((m, p), dtype=float)
    A_full = (X.T * w_main) @ X
    B_full = (X.T * w_main) @ Y
    systems = sorted(pd.unique(np.asarray(country_codes).astype(str)))

    # Independent systems: apply 80 replicates to one system at a time.
    for k, country in enumerate(systems, start=1):
        selected = np.asarray(country_codes).astype(str) == country
        Xc, Yc = X[selected], Y[selected]
        wc, wc_rep = w_main[selected], w_rep[selected, :]
        for r in range(R):
            delta = wc_rep[:, r] - wc
            A_rep = A_full + (Xc.T * delta) @ Xc
            B_rep = B_full + (Xc.T * delta) @ Yc
            beta_rep = np.linalg.solve(A_rep, B_rep)
            difference = (beta_rep - full_beta).T  # PVs × predictors
            sample_var_by_pv += 0.05 * (difference ** 2)
        print(f"{label}: BRR contributions finished for {k}/{len(systems)} systems")

    pv_beta = full_beta.T  # PVs × predictors
    pooled_beta = pv_beta.mean(axis=0)
    sampling_variance = sample_var_by_pv.mean(axis=0)
    pv_variance = (1 + 1/m) * pv_beta.var(axis=0, ddof=1)
    total_variance = sampling_variance + pv_variance
    standard_error = np.sqrt(total_variance)

    # Normal approximation; final reporting should mention this approximation.
    z = np.divide(pooled_beta, standard_error,
                  out=np.full_like(pooled_beta, np.nan), where=standard_error > 0)
    p_approx = np.array([
        math.erfc(abs(float(v))/np.sqrt(2)) if np.isfinite(v) else np.nan
        for v in z
    ])
    z_crit = 1.959963984540054

    predictions = X @ full_beta
    ybar = np.average(Y, axis=0, weights=w_main)
    sse = np.sum(w_main[:, None] * (Y - predictions) ** 2, axis=0)
    sst = np.sum(w_main[:, None] * (Y - ybar) ** 2, axis=0)
    weighted_pv_r2 = 1 - sse / sst

    table = pd.DataFrame({
        "term": term_names,
        "coefficient": pooled_beta,
        "se_total": standard_error,
        "ci_low": pooled_beta - z_crit * standard_error,
        "ci_high": pooled_beta + z_crit * standard_error,
        "p_approx": p_approx,
        "sampling_variance": sampling_variance,
        "plausible_variance": pv_variance,
        "model": label,
    })
    info = {
        "model": label,
        "n_students": int(X.shape[0]),
        "n_terms": int(X.shape[1]),
        "n_systems": len(systems),
        "n_pvs": m,
        "n_replicates_per_system": R,
        "mean_weighted_r2": float(weighted_pv_r2.mean()),
        "elapsed_seconds": round(time.perf_counter() - tic, 2),
        "weighting_strategy": WEIGHTING,
    }
    return table, info

# Quick unit test for the weighted least-squares solver.
test_X = np.array([[1., 0.], [1., 1.], [1., 2.], [1., 3.]])
test_Y = np.array([[2.], [5.], [8.], [11.]])
assert np.allclose(weighted_coefficients(test_X, test_Y, np.ones(4)).ravel(), [2., 3.])
print("Weighted least-squares solver passed the test.")


Weighted least-squares solver passed the test.


PISA provides 10 possible mathematics scores for each student. We run the regression separately for each plausible value and then combine the results.

We also use all 80 replicate weights to estimate sampling uncertainty. This helps us calculate more suitable standard errors and confidence intervals.

In [7]:
rq1, rq1_info = survey_pv_regression(
    X1, terms1, Y, w_main, w_rep, work["CNT"].to_numpy(), "RQ1_baseline"
)
rq2, rq2_info = survey_pv_regression(
    X2, terms2, Y, w_main, w_rep, work["CNT"].to_numpy(), "RQ2_ESCS_adjusted"
)
main_terms = BASE_TERMS + ["ESCS"]
display(pd.concat([rq1, rq2]).query("term in @main_terms").round(4))
display(pd.DataFrame([rq1_info, rq2_info]).round(3))


RQ1_baseline: BRR contributions finished for 1/7 systems
RQ1_baseline: BRR contributions finished for 2/7 systems
RQ1_baseline: BRR contributions finished for 3/7 systems
RQ1_baseline: BRR contributions finished for 4/7 systems
RQ1_baseline: BRR contributions finished for 5/7 systems
RQ1_baseline: BRR contributions finished for 6/7 systems
RQ1_baseline: BRR contributions finished for 7/7 systems
RQ2_ESCS_adjusted: BRR contributions finished for 1/7 systems
RQ2_ESCS_adjusted: BRR contributions finished for 2/7 systems
RQ2_ESCS_adjusted: BRR contributions finished for 3/7 systems
RQ2_ESCS_adjusted: BRR contributions finished for 4/7 systems
RQ2_ESCS_adjusted: BRR contributions finished for 5/7 systems
RQ2_ESCS_adjusted: BRR contributions finished for 6/7 systems
RQ2_ESCS_adjusted: BRR contributions finished for 7/7 systems


,term,coefficient,se_total,ci_low,ci_high,p_approx,sampling_variance,plausible_variance,model
1,EDUSHORT,-4.0508,1.5678,-7.1237,-0.9780,0.0098,2.4468,0.0112,RQ1_baseline
2,CLSIZE,1.5315,0.3006,0.9424,2.1207,0.0000,0.0894,0.0009,RQ1_baseline
3,PROATCE_10PP,0.5751,0.4437,-0.2946,1.4448,0.1949,0.1926,0.0043,RQ1_baseline
1,EDUSHORT,-1.9554,1.2018,-4.3109,0.4001,0.1037,1.4325,0.0118,RQ2_ESCS_adjusted
2,CLSIZE,1.0168,0.2105,0.6041,1.4295,0.0000,0.0435,0.0008,RQ2_ESCS_adjusted
3,PROATCE_10PP,0.4064,0.3486,-0.2768,1.0897,0.2437,0.1172,0.0043,RQ2_ESCS_adjusted
4,ESCS,37.3794,0.7413,35.9264,38.8324,0.0000,0.5034,0.0462,RQ2_ESCS_adjusted


,model,n_students,n_terms,n_systems,n_pvs,n_replicates_per_system,mean_weighted_r2,elapsed_seconds,weighting_strategy
0,RQ1_baseline,70517,10,7,10,80,0.059,0.32,equal_country
1,RQ2_ESCS_adjusted,70517,11,7,10,80,0.194,0.35,equal_country


RQ1 examines the three school factors without controlling for family background.

RQ2 adds ESCS while keeping the other variables and students the same. This allows us to compare how the school-factor coefficients change.

In [8]:
a = rq1.set_index("term")
b = rq2.set_index("term")
comparison = pd.DataFrame(index=BASE_TERMS)
comparison["rq1_beta"] = a.loc[BASE_TERMS, "coefficient"]
comparison["rq1_se"] = a.loc[BASE_TERMS, "se_total"]
comparison["rq2_beta"] = b.loc[BASE_TERMS, "coefficient"]
comparison["rq2_se"] = b.loc[BASE_TERMS, "se_total"]
comparison["difference_rq2_minus_rq1"] = comparison["rq2_beta"] - comparison["rq1_beta"]
comparison["attenuation_pct_abs"] = 100 * (
    abs(comparison["rq1_beta"]) - abs(comparison["rq2_beta"])
) / abs(comparison["rq1_beta"]).replace(0, np.nan)
comparison.index.name = "term"
print("Comparison of school-factor coefficients (same sample and weighting):")
display(comparison.round(3))
print("Model fit (mean of 10 weighted-PV R² values):")
print(f"RQ1 R²: {rq1_info['mean_weighted_r2']:.4f}")
print(f"RQ2 R²: {rq2_info['mean_weighted_r2']:.4f}")

Comparison of school-factor coefficients (same sample and weighting):
Model fit (mean of 10 weighted-PV R² values):
RQ1 R²: 0.0591
RQ2 R²: 0.1938


,rq1_beta,rq1_se,rq2_beta,rq2_se,difference_rq2_minus_rq1,attenuation_pct_abs
term,,,,,,
EDUSHORT,-4.051,1.568,-1.955,1.202,2.095,51.728
CLSIZE,1.532,0.301,1.017,0.211,-0.515,33.608
PROATCE_10PP,0.575,0.444,0.406,0.349,-0.169,29.334


After adding ESCS, all three school-factor coefficients become smaller in absolute terms.

| Factor | RQ1 | RQ2 (with ESCS) | Reduction |
|---|---:|---:|---:|
| Material shortages (per index point) | −4.05 | −1.96 | 52% |
| Class size (per student) | +1.53 | +1.02 | 34% |
| Fully certified teachers (per +10 pp) | +0.58 | +0.41 | 29% |

The model's explained variation increases from 5.9% to 19.4%, and ESCS alone carries a coefficient of about 37 points per index unit.

Statistical significance also changes. Class size is significant in both models. Material shortages are significant in RQ1 (p = 0.010) but not once ESCS is included (p = 0.10, 95% CI −4.3 to 0.4). Teacher certification is not significant in either model (RQ2 95% CI −0.28 to 1.09).

In [9]:
def weighted_sd(v, w):
    mu = np.average(v, weights=w)
    return np.sqrt(np.average((v - mu) ** 2, weights=w))

labels = {
    "EDUSHORT": "Education material shortage",
    "CLSIZE": "Class size",
    "PROATCE_10PP": "Fully certified teachers",
}
fig, ax = plt.subplots(figsize=(9, 4.6))
for i, term in enumerate(BASE_TERMS):
    sd_x = weighted_sd(work[term].to_numpy(float), w_main)
    for model_table, shift, tag, marker in [
        (rq1, -0.12, "RQ1: baseline", "o"),
        (rq2, 0.12, "RQ2: + ESCS", "s"),
    ]:
        row = model_table.set_index("term").loc[term]
        beta = row["coefficient"] * sd_x
        low, high = row["ci_low"] * sd_x, row["ci_high"] * sd_x
        ax.errorbar(
            beta, i + shift, xerr=[[beta-low], [high-beta]],
            fmt=marker, color=("C0" if tag.startswith("RQ1") else "C1"), capsize=3, label=tag if i == 0 else None,
        )
ax.axvline(0, linestyle="--", linewidth=1)
ax.set_yticks(range(len(BASE_TERMS)))
ax.set_yticklabels([labels[t] for t in BASE_TERMS])
ax.invert_yaxis()
ax.set_xlabel("Mathematics score points per 1 weighted SD in predictor (95% CI)")
ax.set_title("PISA 2022: associations before and after ESCS adjustment")
ax.legend()
fig.tight_layout()

out_dir = PROJECT_ROOT / "results/revised/RQ1_RQ2"
out_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir / "rq1_rq2_standardized_coefficients.png", dpi=180, bbox_inches="tight")
plt.show()

The graph compares the three school factors, expressed per one standard deviation, before and after adding ESCS.

Educational-material shortages have a negative association with mathematics scores. Class size and teacher certification have positive associations. All three become weaker after controlling for ESCS. Only class size has a confidence interval that excludes zero in both models.

In [10]:

# HELPER FUNCTIONS

from pathlib import Path
import numpy as np
import pandas as pd
import math

# Retain the original rows corresponding to the modelling sample
raw_complete = df.loc[complete].reset_index(drop=True)

assert len(raw_complete) == len(work)
assert np.array_equal(
    raw_complete["CNT"].astype(str).to_numpy(),
    work["CNT"].astype(str).to_numpy()
)

def build_local_design(frame, terms):
    """Create regression predictors with country fixed effects."""

    country_dummies = pd.get_dummies(
        frame["CNT"].astype(str),
        prefix="CNT",
        drop_first=True,
        dtype=float
    ).reset_index(drop=True)

    X_df = pd.concat(
        [frame[terms].reset_index(drop=True).astype(float),
         country_dummies],
        axis=1
    )

    X_df.insert(0, "Intercept", 1.0)

    X = X_df.to_numpy(dtype=float)

    if not np.isfinite(X).all():
        raise ValueError("Missing or infinite regression values.")

    if np.linalg.matrix_rank(X) < X.shape[1]:
        raise ValueError("Collinear predictors detected.")

    return X, X_df.columns.tolist()


def sample_weights(frame, strategy="equal_country"):
    """Prepare final and replicate weights consistently."""

    w = frame["W_FSTUWT"].to_numpy(dtype=float)
    wr = frame[REP_COLS].to_numpy(dtype=float)

    if strategy == "equal_country":
        totals = frame.groupby("CNT")[
            "W_FSTUWT"
        ].transform("sum").to_numpy(dtype=float)

        factor = 1.0 / totals

    elif strategy == "population":
        factor = np.ones(len(frame))

    else:
        raise ValueError("Unknown weighting strategy.")

    w = w * factor
    wr = wr * factor[:, None]

    if not np.isfinite(w).all() or not np.isfinite(wr).all():
        raise ValueError("Invalid survey weights.")

    if (w <= 0).any() or (wr <= 0).any():
        raise ValueError("Non-positive survey weights.")

    return w, wr


out_dir = PROJECT_ROOT / "results/revised/RQ1_RQ2"
out_dir.mkdir(parents=True, exist_ok=True)

print("Helper functions created successfully.")


Helper functions created successfully.


The following functions help us repeat the regressions using different samples and weights. This allows us to test whether the main results are reliable.

In [11]:

# COUNTRY WEIGHTING SENSITIVITY

# Original PISA population-oriented weights
w_population, wr_population = sample_weights(
    work,
    strategy="population"
)

# Run RQ2 with original survey weights
rq2_population, rq2_pop_info = survey_pv_regression(
    X2,
    terms2,
    Y,
    w_population,
    wr_population,
    work["CNT"].to_numpy(),
    "RQ2_population_weights"
)

# Compare coefficients
equal_results = rq2.set_index("term")
pop_results = rq2_population.set_index("term")

selected = BASE_TERMS + ["ESCS"]

weight_comparison = pd.DataFrame({
    "Variable": selected,
    "Equal_country_beta": equal_results.loc[
        selected, "coefficient"
    ].to_numpy(),
    "Equal_country_SE": equal_results.loc[
        selected, "se_total"
    ].to_numpy(),
    "Original_weight_beta": pop_results.loc[
        selected, "coefficient"
    ].to_numpy(),
    "Original_weight_SE": pop_results.loc[
        selected, "se_total"
    ].to_numpy()
})

weight_comparison["Difference"] = (
    weight_comparison["Original_weight_beta"] -
    weight_comparison["Equal_country_beta"]
)

print("RQ2 weighting comparison:")
display(weight_comparison.round(3))

# Compare country shares
equal_shares = pd.Series(
    w_main
).groupby(work["CNT"]).sum()

population_shares = pd.Series(
    w_population
).groupby(work["CNT"]).sum()

country_weight_shares = pd.DataFrame({
    "Equal_country_%": (
        equal_shares / equal_shares.sum() * 100
    ),
    "Original_weights_%": (
        population_shares / population_shares.sum() * 100
    )
})

print("\nCountry shares:")
display(country_weight_shares.round(2))

# Save results
weight_comparison.to_csv(
    out_dir / "RQ2_weighting_sensitivity.csv",
    index=False
)

rq2_population.to_csv(
    out_dir / "RQ2_population_weights_coefficients.csv",
    index=False
)

print("\nSection 11 completed successfully.")


RQ2_population_weights: BRR contributions finished for 1/7 systems
RQ2_population_weights: BRR contributions finished for 2/7 systems
RQ2_population_weights: BRR contributions finished for 3/7 systems
RQ2_population_weights: BRR contributions finished for 4/7 systems
RQ2_population_weights: BRR contributions finished for 5/7 systems
RQ2_population_weights: BRR contributions finished for 6/7 systems
RQ2_population_weights: BRR contributions finished for 7/7 systems
RQ2 weighting comparison:

Country shares:

Section 11 completed successfully.


,Variable,Equal_country_beta,Equal_country_SE,Original_weight_beta,Original_weight_SE,Difference
0,EDUSHORT,-1.955,1.202,-3.156,1.385,-1.201
1,CLSIZE,1.017,0.211,1.734,0.243,0.717
2,PROATCE_10PP,0.406,0.349,0.860,0.424,0.454
3,ESCS,37.379,0.741,37.396,0.726,0.016


,Equal_country_%,Original_weights_%
CNT,,
DEU,14.29,23.19
ESP,14.29,15.62
EST,14.29,0.45
FRA,14.29,26.59
ITA,14.29,16.89
POL,14.29,11.62
QUR,14.29,5.64


We repeat RQ2 using the original PISA survey weights instead of equal-country weights.

The directions stay the same, but the sizes change by more than their standard errors. The shortage coefficient moves from −1.96 to −3.16, class size from 1.02 to 1.73, and teacher certification from 0.41 to 0.86. With the original weights, shortages (z ≈ 2.3) and teacher certification (z ≈ 2.0) become statistically significant. The original weights give Germany and France about half of the total weight (23% and 27%), while Estonia gets under 1%.

The ESCS coefficient is almost unchanged (37.4 against 37.4). So the ESCS result is robust, but the school-factor results for shortages and certification depend on the weighting choice.

In [12]:

# IMPUTATION SENSITIVITY

def imputed_as_bool(series):
    """Convert common imputation flags to True/False."""

    if pd.api.types.is_bool_dtype(series):
        if series.isna().any():
            raise ValueError(f"Missing flags in {series.name}")
        return series.to_numpy(dtype=bool)

    numeric = pd.to_numeric(series, errors="coerce")

    if numeric.notna().all() and numeric.isin([0, 1]).all():
        return numeric.to_numpy(dtype=int) == 1

    mapping = {
        "true": True, "false": False,
        "yes": True, "no": False,
        "imputed": True, "observed": False,
        "1": True, "0": False
    }

    tokens = series.astype("string").str.strip().str.lower()
    decoded = tokens.map(mapping)

    if decoded.isna().any():
        raise ValueError(
            f"Unknown imputation values in {series.name}"
        )

    return decoded.to_numpy(dtype=bool)


flags = [
    f"{v}_imputed"
    for v in ["ESCS", "EDUSHORT", "CLSIZE", "PROATCE"]
]

available_flags = [
    flag for flag in flags
    if flag in raw_complete.columns
]

missing_flags = [
    flag for flag in flags
    if flag not in available_flags
]

print("Available flags:", available_flags)
print("Unavailable flags:", missing_flags)

imputed_comparison = None

if not available_flags:
    print("No imputation flags available.")

else:
    any_imputed = np.zeros(len(work), dtype=bool)

    for flag in available_flags:
        coded = imputed_as_bool(raw_complete[flag])
        any_imputed |= coded

        print(
            flag,
            "imputed percentage:",
            round(coded.mean() * 100, 2)
        )

    observed_only = (
        work.loc[~any_imputed]
        .reset_index(drop=True)
    )

    print("\nOriginal sample:", len(work))
    print("Sample without flagged imputations:", len(observed_only))

    if len(observed_only) < 200 or observed_only["CNT"].nunique() < 2:
        print("Too few observations for this sensitivity test.")

    else:
        X_observed, terms_observed = build_local_design(
            observed_only,
            BASE_TERMS + ["ESCS"]
        )

        Y_observed = observed_only[
            PV_COLS
        ].to_numpy(dtype=float)

        w_observed, wr_observed = sample_weights(
            observed_only,
            WEIGHTING
        )

        rq2_nonimputed, observed_info = survey_pv_regression(
            X_observed,
            terms_observed,
            Y_observed,
            w_observed,
            wr_observed,
            observed_only["CNT"].to_numpy(),
            "RQ2_no_flagged_imputation"
        )

        all_results = rq2.set_index("term")
        observed_results = rq2_nonimputed.set_index("term")

        imputed_comparison = pd.DataFrame({
            "Variable": selected,

            "All_students_beta": all_results.loc[
                selected, "coefficient"
            ].to_numpy(),

            "Nonimputed_beta": observed_results.loc[
                selected, "coefficient"
            ].to_numpy(),

            "Nonimputed_SE": observed_results.loc[
                selected, "se_total"
            ].to_numpy()
        })

        imputed_comparison["Difference"] = (
            imputed_comparison["Nonimputed_beta"] -
            imputed_comparison["All_students_beta"]
        )

        display(imputed_comparison.round(3))

        imputed_comparison.to_csv(
            out_dir / "RQ2_imputation_sensitivity.csv",
            index=False
        )


Available flags: ['ESCS_imputed', 'EDUSHORT_imputed', 'CLSIZE_imputed', 'PROATCE_imputed']
Unavailable flags: []
ESCS_imputed imputed percentage: 3.82
EDUSHORT_imputed imputed percentage: 8.96
CLSIZE_imputed imputed percentage: 8.85
PROATCE_imputed imputed percentage: 15.33

Original sample: 70517
Sample without flagged imputations: 56287
RQ2_no_flagged_imputation: BRR contributions finished for 1/7 systems
RQ2_no_flagged_imputation: BRR contributions finished for 2/7 systems
RQ2_no_flagged_imputation: BRR contributions finished for 3/7 systems
RQ2_no_flagged_imputation: BRR contributions finished for 4/7 systems
RQ2_no_flagged_imputation: BRR contributions finished for 5/7 systems
RQ2_no_flagged_imputation: BRR contributions finished for 6/7 systems
RQ2_no_flagged_imputation: BRR contributions finished for 7/7 systems


,Variable,All_students_beta,Nonimputed_beta,Nonimputed_SE,Difference
0,EDUSHORT,-1.955,-2.135,1.228,-0.179
1,CLSIZE,1.017,1.108,0.218,0.092
2,PROATCE_10PP,0.406,0.436,0.343,0.029
3,ESCS,37.379,37.167,0.778,-0.212


Some values were filled during data cleaning (up to 15.3% of PROATCE and 9.0% of EDUSHORT). We repeat RQ2 without students whose predictors were flagged as imputed.

The sample decreases from 70,517 to 56,287 students. The coefficients change little (EDUSHORT −2.14 vs −1.96, CLSIZE 1.11 vs 1.02, PROATCE 0.44 vs 0.41, ESCS 37.2 vs 37.4), and the significance pattern is the same. This complete-case sensitivity gives similar coefficients; it does not establish that missingness is ignorable.

Coefficients are in different units, so we also express each association per one weighted standard deviation of the predictor.

In [13]:
std_rows = []
for t in BASE_TERMS:
    sd = weighted_sd(work[t].to_numpy(float), w_main)
    for m, tab in [("RQ1", rq1), ("RQ2", rq2)]:
        r = tab.set_index("term").loc[t]
        std_rows.append({"model": m, "factor": t, "sd_of_predictor": sd,
                         "effect_per_1SD": r["coefficient"] * sd})
std_table = (pd.DataFrame(std_rows)
             .pivot(index="factor", columns="model", values="effect_per_1SD")[["RQ1", "RQ2"]])
display(std_table.round(2))

model,RQ1,RQ2
factor,,
CLSIZE,10.73,7.13
EDUSHORT,-4.05,-1.96
PROATCE_10PP,1.71,1.21


In [14]:
# SENSITIVITY: drop Ukraine (QUR) -> 6 systems
no_ukr = work[work["CNT"] != "QUR"].reset_index(drop=True)
X_nu, terms_nu = build_local_design(no_ukr, BASE_TERMS + ["ESCS"])
Y_nu = no_ukr[PV_COLS].to_numpy(dtype=float)
w_nu, wr_nu = sample_weights(no_ukr, WEIGHTING)

rq2_nu, info_nu = survey_pv_regression(
    X_nu, terms_nu, Y_nu, w_nu, wr_nu, no_ukr["CNT"].to_numpy(), "RQ2_without_Ukraine"
)
a = rq2.set_index("term"); b = rq2_nu.set_index("term")
ukr_comparison = pd.DataFrame({
    "Variable": selected,
    "All_7_beta": a.loc[selected, "coefficient"].to_numpy(),
    "Without_UKR_beta": b.loc[selected, "coefficient"].to_numpy(),
    "Without_UKR_SE": b.loc[selected, "se_total"].to_numpy(),
})
ukr_comparison["Difference"] = ukr_comparison["Without_UKR_beta"] - ukr_comparison["All_7_beta"]
display(ukr_comparison.round(3))

RQ2_without_Ukraine: BRR contributions finished for 1/6 systems
RQ2_without_Ukraine: BRR contributions finished for 2/6 systems
RQ2_without_Ukraine: BRR contributions finished for 3/6 systems
RQ2_without_Ukraine: BRR contributions finished for 4/6 systems
RQ2_without_Ukraine: BRR contributions finished for 5/6 systems
RQ2_without_Ukraine: BRR contributions finished for 6/6 systems


,Variable,All_7_beta,Without_UKR_beta,Without_UKR_SE,Difference
0,EDUSHORT,-1.955,-2.471,1.146,-0.515
1,CLSIZE,1.017,1.436,0.216,0.419
2,PROATCE_10PP,0.406,0.925,0.398,0.518
3,ESCS,37.379,37.131,0.686,-0.249


Ukraine's data cover 18 of 27 regions and may be affected by the war. We therefore repeat RQ2 on the six remaining systems.

Without Ukraine, the shortage coefficient is −2.47 (z ≈ 2.2) and teacher certification is +0.93 (z ≈ 2.3), so both become statistically significant. Class size rises from 1.02 to 1.44, and ESCS is unchanged (37.1). Ukraine therefore weakens the shortage and certification associations: its own coefficients are in the opposite direction (see the per-country table below). Their signs remain positive when QUR is excluded in this specification; this does not establish robustness to other changes.

In [15]:
rows = []
names_c = ["Intercept"] + BASE_TERMS + ["ESCS"]
for c in sorted(work["CNT"].unique()):
    sub = work[work["CNT"] == c].reset_index(drop=True)
    Xc = np.column_stack([np.ones(len(sub)), sub[BASE_TERMS + ["ESCS"]].to_numpy(float)])
    wc, wrc = sample_weights(sub, "population")   # single country: scale is irrelevant
    tab, _ = survey_pv_regression(Xc, names_c, sub[PV_COLS].to_numpy(float),
                                  wc, wrc, sub["CNT"].to_numpy(), f"RQ2_{c}")
    tab["country"] = c
    rows.append(tab)

per_country = pd.concat(rows)
per_country = per_country[per_country["term"].isin(BASE_TERMS)]
display(per_country.pivot(index="country", columns="term", values="coefficient").round(2))
display(per_country.pivot(index="country", columns="term", values="se_total").round(2))

RQ2_DEU: BRR contributions finished for 1/1 systems
RQ2_ESP: BRR contributions finished for 1/1 systems
RQ2_EST: BRR contributions finished for 1/1 systems
RQ2_FRA: BRR contributions finished for 1/1 systems
RQ2_ITA: BRR contributions finished for 1/1 systems
RQ2_POL: BRR contributions finished for 1/1 systems
RQ2_QUR: BRR contributions finished for 1/1 systems


term,CLSIZE,EDUSHORT,PROATCE_10PP
country,,,
DEU,4.86,-5.84,1.58
ESP,0.19,-3.15,0.07
EST,0.63,-1.75,0.63
FRA,5.72,-1.71,1.24
ITA,0.57,-6.68,1.32
POL,1.14,4.90,0.63
QUR,0.18,0.79,-0.71


term,CLSIZE,EDUSHORT,PROATCE_10PP
country,,,
DEU,0.77,3.35,1.50
ESP,0.11,1.20,0.39
EST,0.31,1.72,0.67
FRA,0.53,2.90,1.07
ITA,0.55,3.77,1.12
POL,0.51,2.86,1.05
QUR,0.39,4.69,0.68


To examine heterogeneity, we fit RQ2 separately in each system. This is not a leave-one-country-out test.

The estimates differ a lot between countries. Class size is strongly positive in Germany (4.86) and France (5.72) but close to zero in Spain (0.19) and Ukraine (0.18). The shortage coefficient is negative in five systems but positive in Poland (+4.90) and Ukraine (+0.79), and its standard errors are large (1.2 to 4.7). Teacher certification is positive in six systems but small and imprecise in each.

The pooled results should therefore be read as an average of unequal country patterns, not as a rule that holds everywhere. The class-size association is driven mainly by Germany and France.

Class size is positive, which is unexpected if smaller classes help. One explanation is school selection: weaker students may be placed in smaller classes or in schools with different intakes. To test this, we add the average socioeconomic status of the school (SCH_ESCS) to the RQ2 model.

In [16]:
work["SCH_ESCS"] = raw_complete["SCH_ESCS"].to_numpy(float)
X3, terms3 = build_local_design(work, BASE_TERMS + ["ESCS", "SCH_ESCS"])
rq2b, info2b = survey_pv_regression(X3, terms3, Y, w_main, w_rep,
                                    work["CNT"].to_numpy(), "RQ2b_school_ESCS")
display(rq2b[rq2b["term"].isin(BASE_TERMS + ["ESCS", "SCH_ESCS"])].round(3))

RQ2b_school_ESCS: BRR contributions finished for 1/7 systems
RQ2b_school_ESCS: BRR contributions finished for 2/7 systems
RQ2b_school_ESCS: BRR contributions finished for 3/7 systems
RQ2b_school_ESCS: BRR contributions finished for 4/7 systems
RQ2b_school_ESCS: BRR contributions finished for 5/7 systems
RQ2b_school_ESCS: BRR contributions finished for 6/7 systems
RQ2b_school_ESCS: BRR contributions finished for 7/7 systems


,term,coefficient,se_total,ci_low,ci_high,p_approx,sampling_variance,plausible_variance,model
1,EDUSHORT,1.097,0.929,-0.725,2.918,0.238,0.851,0.013,RQ2b_school_ESCS
2,CLSIZE,0.241,0.132,-0.018,0.500,0.069,0.016,0.001,RQ2b_school_ESCS
3,PROATCE_10PP,0.176,0.309,-0.429,0.781,0.568,0.091,0.004,RQ2b_school_ESCS
4,ESCS,17.994,0.625,16.769,19.220,0.000,0.314,0.076,RQ2b_school_ESCS
5,SCH_ESCS,71.765,1.780,68.277,75.254,0.000,2.957,0.211,RQ2b_school_ESCS


After adding school-average ESCS, the class-size coefficient decreases from 1.02 to 0.24 and is no longer statistically significant (p = 0.069).

The other school factors are also not statistically significant. This suggests that school socioeconomic composition may explain some of the relationships seen in the earlier models.

In [17]:

# PAIRED COEFFICIENT CHANGE TEST

def paired_coefficient_changes(
    Xa, names_a, Xb, names_b,
    Y, weights, rep_weights,
    countries, terms_to_compare
):
    """Test paired changes using all PVs and Fay BRR weights."""

    indices_a = [
        names_a.index(t) for t in terms_to_compare
    ]

    indices_b = [
        names_b.index(t) for t in terms_to_compare
    ]

    beta_a = weighted_coefficients(Xa, Y, weights)
    beta_b = weighted_coefficients(Xb, Y, weights)

    main_diff = (
        beta_b[indices_b, :] -
        beta_a[indices_a, :]
    ).T

    sampling_by_pv = np.zeros_like(main_diff)

    A_a = (Xa.T * weights) @ Xa
    B_a = (Xa.T * weights) @ Y

    A_b = (Xb.T * weights) @ Xb
    B_b = (Xb.T * weights) @ Y

    codes = np.asarray(countries).astype(str)

    for country in sorted(np.unique(codes)):

        mask = codes == country

        xa = Xa[mask]
        xb = Xb[mask]
        ys = Y[mask]

        w0 = weights[mask]
        wr0 = rep_weights[mask]

        for r in range(wr0.shape[1]):

            delta = wr0[:, r] - w0

            rep_a = np.linalg.solve(
                A_a + (xa.T * delta) @ xa,
                B_a + (xa.T * delta) @ ys
            )

            rep_b = np.linalg.solve(
                A_b + (xb.T * delta) @ xb,
                B_b + (xb.T * delta) @ ys
            )

            rep_diff = (
                rep_b[indices_b, :] -
                rep_a[indices_a, :]
            ).T

            # Fay BRR variance contribution
            sampling_by_pv += (
                0.05 * (rep_diff - main_diff) ** 2
            )

    # Pool the 10 plausible values
    difference = main_diff.mean(axis=0)

    sampling_variance = sampling_by_pv.mean(axis=0)

    plausible_variance = (
        (1 + 1 / Y.shape[1]) *
        main_diff.var(axis=0, ddof=1)
    )

    se = np.sqrt(
        sampling_variance + plausible_variance
    )

    p_values = np.array([
        math.erfc(abs(float(d / s)) / np.sqrt(2))
        if s > 0 else np.nan
        for d, s in zip(difference, se)
    ])

    return pd.DataFrame({
        "Variable": terms_to_compare,
        "RQ2_minus_RQ1": difference,
        "Standard_error": se,
        "CI_low": difference - 1.95996 * se,
        "CI_high": difference + 1.95996 * se,
        "p_value_approx": p_values
    })


coefficient_change_tests = paired_coefficient_changes(
    X1,
    terms1,
    X2,
    terms2,
    Y,
    w_main,
    w_rep,
    work["CNT"].to_numpy(),
    BASE_TERMS
)

print("Statistical tests of coefficient changes:")
display(coefficient_change_tests.round(4))

coefficient_change_tests.to_csv(
    out_dir / "RQ1_RQ2_coefficient_change_tests.csv",
    index=False
)


Statistical tests of coefficient changes:


,Variable,RQ2_minus_RQ1,Standard_error,CI_low,CI_high,p_value_approx
0,EDUSHORT,2.0954,0.5059,1.1038,3.0871,0.0000
1,CLSIZE,-0.5147,0.1065,-0.7235,-0.3060,0.0000
2,PROATCE_10PP,-0.1687,0.1567,-0.4759,0.1385,0.2818


We test whether the differences between RQ1 and RQ2 are statistically significant, using the same plausible values and replicate weights.

The change for material shortages (+2.10, SE 0.51) and for class size (−0.51, SE 0.11) is significant at the 5% level. The change for teacher certification (−0.17, SE 0.16, p = 0.28) is not. Adjusting for family background substantially attenuates the shortage and class-size associations, but we cannot say that it changes the certification coefficient

In [18]:

# FINAL ROBUSTNESS FIGURES

import matplotlib.pyplot as plt

labels_report = {
    "EDUSHORT": "Educational-material shortages",
    "CLSIZE": "Class size",
    "PROATCE_10PP": "Fully certified teachers (+10 pp)",
    "ESCS": "Student socioeconomic status"
}

def comparison_plot(models, variables, title, filename):
    """Plot coefficients and 95% confidence intervals."""

    fig, ax = plt.subplots(figsize=(10, 5))

    offsets = np.linspace(-0.15, 0.15, len(models))
    symbols = ["o", "s", "^", "D"]

    for i, term in enumerate(variables):

        for j, (table, label) in enumerate(models):

            result = table.set_index("term").loc[term]

            beta = result["coefficient"]

            lower_error = beta - result["ci_low"]
            upper_error = result["ci_high"] - beta

            ax.errorbar(
                beta,
                i + offsets[j],
                xerr=[[lower_error], [upper_error]],
                fmt=symbols[j % len(symbols)],
                color=f"C{j}",
                capsize=3,
                label=label if i == 0 else None
            )

    ax.axvline(
        0,
        color="gray",
        linestyle="--",
        linewidth=1
    )

    ax.set_yticks(
        range(len(variables)),
        [labels_report[t] for t in variables]
    )

    ax.invert_yaxis()

    ax.set_xlabel(
        "Mathematics score points per predictor unit (95% CI)"
    )

    ax.set_title(title)
    ax.legend()

    plt.tight_layout()

    plt.savefig(
        out_dir / filename,
        dpi=200,
        bbox_inches="tight"
    )

    plt.show()


# Figure 1: Equal-country vs original weights
comparison_plot([(rq2, "Equal-country weighting"), (rq2_population, "Original PISA weighting")],
                BASE_TERMS, "RQ2: Sensitivity to survey weighting", "rq2_weighting_sensitivity.png")



The final graph compares the three school factors under the two weighting methods.

The signs are the same under both weightings. The estimates are larger under the original PISA weights, and the intervals for shortages and teacher certification move away from zero. The conclusion for these two factors is sensitive to how countries are weighted, whereas class size is positive and significant under both.

In [19]:

# RQ1 AND RQ2

from pathlib import Path

results_dir = out_dir
results_dir.mkdir(parents=True, exist_ok=True)

# 1. Main regression results
regression_results = pd.concat(
    [rq1, rq2],
    ignore_index=True
)

regression_results.to_csv(
    results_dir / "RQ1_RQ2_coefficients.csv",
    index=False
)

# 2. Coefficient comparison
comparison.reset_index().to_csv(
    results_dir / "RQ1_RQ2_comparison.csv",
    index=False
)

# 3. Model fit
model_summary = pd.DataFrame(
    [rq1_info, rq2_info]
)

model_summary.to_csv(
    results_dir / "RQ1_RQ2_model_summary.csv",
    index=False
)

# 4. Statistical tests of coefficient changes
coefficient_change_tests.to_csv(
    results_dir / "RQ1_RQ2_coefficient_change_tests.csv",
    index=False
)

# 5. Weighting sensitivity
weight_comparison.to_csv(
    results_dir / "RQ2_weighting_sensitivity.csv",
    index=False
)

# 6. Imputation sensitivity
if imputed_comparison is not None:
    imputed_comparison.to_csv(
        results_dir / "RQ2_imputation_sensitivity.csv",
        index=False
    )

# 7. Confirm exported files
print("RQ1 AND RQ2 EXPORT COMPLETED")
print("\nResults folder:", results_dir.resolve())

print("\nSaved files:")

for file in sorted(results_dir.iterdir()):
    if file.is_file():
        print("-", file.name)

print("\nMain regression summary:")
display(model_summary[
    ["model", "n_students", "n_systems",
     "n_pvs", "mean_weighted_r2"]
].round(4))


RQ1 AND RQ2 EXPORT COMPLETED

Results folder: /Users/danyloredko/Documents/Codex/2026-11-12/referenced-chatgpt-conversation-this-is-an/work/repo_current/results/revised/RQ1_RQ2

Saved files:
- RQ1_RQ2_coefficient_change_tests.csv
- RQ1_RQ2_coefficients.csv
- RQ1_RQ2_comparison.csv
- RQ1_RQ2_model_summary.csv
- RQ1_RQ2_standardized_coefficients.csv
- RQ2_country_coefficients.csv
- RQ2_imputation_sensitivity.csv
- RQ2_population_weights_coefficients.csv
- RQ2_vs_RQ2b_coefficients.png
- RQ2_weighting_sensitivity.csv
- RQ2_without_QUR_coefficients.csv
- RQ2_without_QUR_comparison.csv
- RQ2b_school_context_coefficients.csv
- rq1_rq2_standardized_coefficients.png
- rq2_weighting_sensitivity.png
- run_manifest.json

Main regression summary:


,model,n_students,n_systems,n_pvs,mean_weighted_r2
0,RQ1_baseline,70517,7,10,0.0591
1,RQ2_ESCS_adjusted,70517,7,10,0.1938


Summary
Socioeconomic background is strongly related to mathematics performance (about 37 points per ESCS unit). After controlling for it, the associations between school factors and performance become smaller: by 52% for material shortages, 34% for class size and 29% for teacher certification.

Of the three factors, only class size is significant in both RQ1 and RQ2, and its coefficient is positive. In RQ2b, which also adjusts for school ESCS, it falls to +0.24 and is not significant at 5% (p = 0.069). The shortage association is significant before the ESCS control but not after it, and teacher certification is not significant in the main models. The shortage and certification results depend on weighting and on whether Ukraine is included, and the country-by-country estimates differ widely.

Limitations
- The analysis shows relationships, not cause and effect, so it cannot prove which investment would improve performance. Selection and school intake are possible explanations for the positive class-size coefficient; these models do not establish its mechanism.
- The seven systems were chosen by the team and are not a representative sample of PISA countries. Ukraine covers 18 of 27 regions and may be affected by the war.
- EDUSHORT and PROATCE are proxies for "equipment" and "teacher qualification". Up to 15.3% of PROATCE values were imputed.
- 2022 response rates were low in several countries because of COVID.
- Normal approximations are used for p-values and confidence intervals.

In [20]:
rq2b.to_csv(out_dir / "RQ2b_school_context_coefficients.csv", index=False)
ukr_comparison.to_csv(out_dir / "RQ2_without_QUR_comparison.csv", index=False)
rq2_nu.to_csv(out_dir / "RQ2_without_QUR_coefficients.csv", index=False)
per_country.to_csv(out_dir / "RQ2_country_coefficients.csv", index=False)
std_table.to_csv(out_dir / "RQ1_RQ2_standardized_coefficients.csv")
comparison_plot([(rq2, "RQ2: student ESCS"), (rq2b, "RQ2b: student + school ESCS")], BASE_TERMS, "School intake adjustment changes the policy associations", "RQ2_vs_RQ2b_coefficients.png")
